In [0]:
USE CATALOG dbw_kroger_insights;

In [0]:
CREATE OR REPLACE TABLE gold.kpi_daily
COMMENT 'Daily eCommerce KPIs for dashboards. One row per calendar day (Oct–Nov 2019). 7-day averages are null until the window is full. is_anomaly_window flags Nov 14–17 (sale event + Nov 15 checkout/tracking incident).'
AS
WITH daily AS (
    SELECT
        d.date,
        d.date_key,
        d.day_name,
        d.week_start_date,
        d.year_month,
        d.is_weekend,
        d.is_black_friday,
        d.date BETWEEN DATE'2019-11-14' AND DATE'2019-11-17'                     AS is_anomaly_window,
        COUNT(s.user_session)                                                     AS sessions,
        COUNT_IF(s.has_purchase)                                                  AS converting_sessions,
        ROUND(COALESCE(SUM(s.revenue), 0), 2)                                     AS revenue
    FROM gold.dim_date d
    LEFT JOIN gold.ec_fact_sessions s
        ON s.session_date = d.date
    GROUP BY d.date, d.date_key, d.day_name, d.week_start_date, d.year_month,
             d.is_weekend, d.is_black_friday
)
SELECT
    *,
    ROUND(100.0 * converting_sessions / NULLIF(sessions, 0), 2)                   AS session_conversion_pct,
    ROUND(revenue / NULLIF(sessions, 0), 2)                                        AS revenue_per_session,
    CASE WHEN COUNT(*) OVER w = 7 THEN ROUND(AVG(sessions) OVER w, 0) END          AS sessions_7d_avg,
    CASE WHEN COUNT(*) OVER w = 7 THEN ROUND(AVG(revenue)  OVER w, 2) END          AS revenue_7d_avg
FROM daily
WINDOW w AS (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW);

In [0]:
   SELECT COUNT(*)                     AS days,
          SUM(sessions)                AS sessions,
          ROUND(SUM(revenue), 2)       AS revenue,
          COUNT_IF(is_anomaly_window)  AS anomaly_days
   FROM gold.kpi_daily;

In [0]:
CREATE OR REPLACE TABLE dbw_kroger_insights.gold.sku_segments 
COMMENT 'SKU-level performance segmentation table for merchandising, pricing, and catalog optimization workflows.'
AS
WITH product_totals AS (
    SELECT 
        product_id,
        SUM(views)                                                             AS views,
        SUM(carts)                                                             AS carts,
        SUM(purchases)                                                         AS purchases,
        ROUND(SUM(revenue), 2)                                                 AS revenue
    FROM dbw_kroger_insights.gold.ec_fact_product_daily
    GROUP BY product_id
),
ranked_products AS (
    SELECT
        product_id,
        views,
        carts,
        purchases,
        revenue,
        ROW_NUMBER() OVER (ORDER BY revenue DESC)                              AS revenue_rank,
        SUM(revenue) OVER (ORDER BY revenue DESC
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)  AS cumulative_revenue,
        SUM(revenue) OVER ()                                                   AS total_revenue
    FROM product_totals
),
with_cum_pct AS (
    SELECT
        product_id,
        views,
        carts,
        purchases,
        revenue,
        revenue_rank,
        -- Cumulative revenue percentage (0.0 to 1.0)
        CASE 
            WHEN total_revenue > 0 THEN cumulative_revenue / total_revenue 
            ELSE 0 
        END                                                                    AS cumulative_revenue_pct
    FROM ranked_products
),
segmented_products AS (
    SELECT
        w.product_id,
        p.category_l1                                                          AS category_l1,
        p.brand,
        ROUND(p.median_price, 2)                                               AS median_price,
        w.views,
        w.purchases,
        ROUND(w.revenue, 2)                                                    AS revenue,
        w.revenue_rank,
        ROUND(w.cumulative_revenue_pct * 100.0, 2)                             AS cumulative_revenue_pct,
        
        -- Segment Rule Evaluation (ordered strictly by priority)
        CASE 
            WHEN w.purchases > 0 AND w.cumulative_revenue_pct <= 0.80 
                THEN 'Hero'
            WHEN w.purchases > 0 AND w.cumulative_revenue_pct > 0.80 
                THEN 'Core'
            WHEN w.purchases = 0 AND w.views >= 1000 
                THEN 'Fix'
            WHEN w.purchases = 0 AND w.views BETWEEN 100 AND 999 
                THEN 'Watch'
            ELSE 'Tail review'
        END                                                                    AS segment
    FROM with_cum_pct w
    LEFT JOIN dbw_kroger_insights.gold.ec_dim_product p
        ON w.product_id = p.product_id
)
SELECT
    product_id,
    category_l1,
    brand,
    median_price,
    views,
    purchases,
    revenue,
    revenue_rank,
    cumulative_revenue_pct,
    segment,
    -- Recommended Action mapping based on segment
    CASE segment
        WHEN 'Hero' THEN 'Protect availability and pricing'
        WHEN 'Core' THEN 'Maintain'
        WHEN 'Fix' THEN 'Review price, stock, and content'
        WHEN 'Watch' THEN 'Investigate by category'
        WHEN 'Tail review' THEN 'Delist or discoverability review'
        ELSE 'Review'
    END                                                                        AS recommended_action
FROM segmented_products;

In [0]:
   SELECT
       segment,
       COUNT(*)                                                     AS skus,
       ROUND(SUM(revenue), 2)                                       AS revenue,
       ROUND(100.0 * SUM(revenue) / SUM(SUM(revenue)) OVER (), 2)   AS revenue_share_pct
   FROM gold.sku_segments
   GROUP BY segment
   ORDER BY revenue DESC;